# Toy models: convergence of the five objectives

Checks that every toy variant converged before its latent geometry is interpreted. It answers whether training stopped on a plateau of the validation loss and how the reconstruction term compares across objectives.

## Introduction

The toy campaign (`kidney-representative-toy-circle`) trains cumulative objectives on one small kidney image with latent width $L = 3$, so that the whole latent space is $S^1$ and can be shown directly. Its figures are used in the conceptual presentation (`assets/documents/prestentations/segmentation_model_full_conceptual`).

### Assumptions

- One model per objective (one repetition). Differences between variants are single realizations, not
  estimates of an expected effect; the toy is illustrative, not a benchmark.
- All variants share the data split, the initialization seed and the training seed, so the only
  difference is the objective.
- Architecture, optimizer, loss parameters and loss weights follow the `real_only` schedule of the
  `20_09_26_metaspace_base_pretrain` campaign (without label-weighted contrastive negatives), including
  its 10 epochs; only `latent_dim` is smaller.
- Ten epochs on the training part of 16 230 pixels are about 2000 optimizer steps; the
  auxiliary terms (weights 0.2, 0.001, 0.002) contribute little to the objective, so differences between
  variants are expected to be small.
- Display categories are built from five selected ions (three spatially disjoint, one overlapping one of
  them, one present almost everywhere) by the rules in `analysis_settings.yaml`; names list the ions.

### Notation

| symbol | meaning |
|---|---|
| $x \in \Delta^{M-1}$ | TIC-normalized binned spectrum, $M = 1364$ bins of $0.55$ m/z on $[200, 950]$ |
| $a \in \mathbb{R}^{L}$ | encoder output before the bottleneck `LayerNorm`, $L = 3$ |
| $u = (a - \mu_a \mathbf 1)/\sigma_a$ | canonical latent, $\mathbf 1^\top u = 0$, $\lVert u \rVert = \sqrt L$ |
| $z = \gamma \odot u + \beta$ | model latent (affine `LayerNorm` output) |
| $s = Q^\top u / \lVert Q^\top u \rVert \in S^1$ | latent coordinates; $Q \in \mathbb{R}^{3 \times 2}$ fixed orthonormal basis of $\mathbf 1^\perp$ |
| $\mathcal M_0, \dots, \mathcal M_4$ | objectives: reconstruction; + head; + head + contrastive; + head + contractive; + head + both |
| $\angle(u, u')$ | angle between canonical latents, in degrees |

## Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from msi_autoencoder_wrapper.analysis.autoencoder.experiments import representative_toy_precompute as toy
from msi_autoencoder_wrapper.visualization import representative_toy as viz

SETTINGS = toy.load_settings(Path("analysis_settings.yaml"))
FIGURES = Path(SETTINGS["repository_root"]) / SETTINGS["figures"]["presentation_directory"]
LABELS = SETTINGS["campaign"]["labels"]
SHOWN = SETTINGS["campaign"]["presentation"]  # variants used on the slides
PREFIX = SETTINGS["figures"]["prefix"]
REGION_COLORS = {rule["name"]: rule["color"] for rule in SETTINGS["regions"]}

classes = toy.load_table(SETTINGS, "dataset", "classes")
pixels = toy.load_table(SETTINGS, "dataset", "pixels")
pixels["region"] = toy.assign_regions(pixels, classes, SETTINGS["regions"])

## Producing the tables this notebook reads

The tables are written by the toy precomputation module; the command below is generated by that module.

In [ ]:
print(toy.run_command(SETTINGS, "training"))

## Training and validation loss per variant

### Methodology

#### Theoretical

The training objective of variant $k$ is $\mathcal L_k = W_1(x, \hat x) + \sum_j w_j \mathcal L_j$ with the variant's auxiliary terms $\mathcal L_j$; the best checkpoint minimizes the validation total loss.

#### Implementation

Per-epoch values are read from each model's `history.json` (written by the library trainer); only numeric metrics of epoch records are kept.

#### Figure descriptions

x = epoch, y = loss (log scale); one panel per variant; lines = recorded loss components.

In [ ]:
history = toy.load_table(SETTINGS, "training", "history")
display(history.groupby("variant").epoch.max().rename("epochs").to_frame())
print(sorted(c for c in history.columns if c not in ("variant", "order", "epoch")))

In [ ]:
metrics = [c for c in history.columns if c not in ("variant", "order", "epoch") and history[c].notna().any()]
with viz.presentation_style():
    figure, axes = plt.subplots(1, history.variant.nunique(), figsize=(22, 4.6), sharey=False)
    for ax, (variant, frame) in zip(axes, history.groupby("variant", sort=False)):
        for metric in metrics:
            values = frame[metric]
            if values.notna().any() and (values > 0).all():
                ax.plot(frame.epoch, values, lw=1.2, label=metric)
        ax.set_yscale("log")
        ax.set_title(LABELS[variant], fontsize=13)
        ax.set_xlabel("epoch")
    axes[0].legend(fontsize=8, frameon=False)
    plt.show()

### Remarks

### Notes

## Results / Summary

### LLM

### Person